In [1]:
import cv2
import numpy as np
from pathlib import Path

# Model files 
MODEL_WEIGHTS = 'frozen_inference_graph.pb'
MODEL_CONFIG  = 'ssd_mobilenet_v2_coco_2018_03_29.pbtxt'

net = cv2.dnn_DetectionModel(MODEL_WEIGHTS, MODEL_CONFIG)
net.setInputSize(320, 320)
net.setInputScale(1.0 / 127.5)
net.setInputMean((127.5, 127.5, 127.5))
net.setInputSwapRB(True)
print("✅ Model loaded")

✅ Model loaded


In [2]:
with open('coco.names', 'rt') as f:
    classNames = f.read().rstrip('\n').split('\n')

person_class_id = classNames.index('person')  # is 0
print(f"'person' class id = {person_class_id}")

'person' class id = 0


In [3]:
cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)  # try index 0 first
ret, frame = cap.read()
cap.release()

print("ret =", ret, "| frame shape =" if ret else "", frame.shape if ret else "")
if ret:
    cv2.imshow("Cam Test – press any key", frame)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

ret = True | frame shape = (480, 640, 3)


In [4]:
import cv2
import numpy as np

#Load network using readNetFromTensorflow (same as test)
net = cv2.dnn.readNetFromTensorflow(
    'frozen_inference_graph.pb',
    'ssd_mobilenet_v2_coco_2018_03_29.pbtxt'
)

cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    h, w = frame.shape[:2]
    blob = cv2.dnn.blobFromImage(frame, size=(300, 300), swapRB=True, crop=False)
    net.setInput(blob)
    detections = net.forward()

    for i in range(detections.shape[2]):
        confidence = detections[0, 0, i, 2]
        if confidence > 0.5:
            class_id = int(detections[0, 0, i, 1])
            if class_id == 1:  # person
                box = detections[0, 0, i, 3:7] * np.array([w, h, w, h])
                (x1, y1, x2, y2) = box.astype(int)
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.putText(frame, f'Person {confidence:.2f}', (x1, y1 - 10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    cv2.imshow('Person Detection', frame)
    if cv2.waitKey(1) == 27:  # ESC to quit
        break

cap.release()
cv2.destroyAllWindows()